# 🧠 Prática 3: Ciência de Dados
### Workshop Dados na Prática · Nerd Data Lab + Programa AI

A análise mostrou **o que aconteceu**. A ciência de dados tenta **prever o que vai acontecer**.

Nossa pergunta agora é: **quais clientes têm mais risco de deixar de comprar?** Com essa resposta, o dono do café pode fazer uma campanha para reconquistá-los antes que seja tarde.

Vamos treinar um modelo de **Machine Learning**: um programa que aprende padrões olhando exemplos do passado.

**Como usar:** Shift + Enter para rodar. Nas células com ✏️, troque os `___`.

In [ ]:
# ⚙️ Configuração: rode esta célula primeiro (Shift + Enter)
import os
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 4)

# De onde vêm os dados:
# 1) Se você subiu a pasta cafe_tambau para o Colab (ícone de pasta à esquerda), ela é usada.
# 2) Se não, os arquivos são baixados deste endereço.
URL_BASE = "https://raw.githubusercontent.com/wuldson-franco/workshop_dados_pratica/main/"
PASTA_LOCAL = "./"

def ler(arquivo, **opcoes):
    caminho = PASTA_LOCAL + arquivo
    if os.path.exists(caminho):
        return pd.read_csv(caminho, **opcoes)
    return pd.read_csv(URL_BASE + arquivo, **opcoes)

print("Tudo pronto! ✅")

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, recall_score

In [ ]:
vendas = ler("dados_tratados/vendas_limpas.csv", parse_dates=["data_hora"])
clientes = ler("dados_tratados/clientes_limpos.csv")
print("Vendas:", vendas.shape, "| Clientes:", clientes.shape)

## 1. Definir o que é "deixar de comprar"

Vamos dividir o tempo em duas partes:

- **Antes** (janeiro a junho): o comportamento do cliente. É daqui que o modelo vai aprender.
- **Depois** (15 de agosto a 30 de setembro): o resultado. Se o cliente comprou no "antes" mas **não apareceu** no "depois", dizemos que ele **sumiu**.

In [ ]:
identificadas = vendas.dropna(subset=["id_cliente"])

antes = identificadas[identificadas["data_hora"] < "2026-07-01"]
depois = identificadas[identificadas["data_hora"] >= "2026-08-15"]

clientes_antes = set(antes["id_cliente"])
clientes_depois = set(depois["id_cliente"])

print("Clientes que compraram de jan a jun:", len(clientes_antes))
print("Desses, quantos sumiram:", len(clientes_antes - clientes_depois))

## 2. Criar as características de cada cliente (features)

O modelo não entende "o João gosta de cappuccino". Ele entende números. Então transformamos o comportamento de cada cliente em colunas:

| Feature | O que significa |
|---|---|
| `pedidos` | Quantos pedidos fez de jan a jun |
| `gasto_total` | Quanto gastou |
| `ticket_medio` | Gasto médio por pedido |
| `pct_cappuccino` | Em quantos % dos pedidos pediu cappuccino |
| `pct_ifood` | Em quantos % dos pedidos usou o iFood |
| `pct_manha` | Em quantos % dos pedidos veio de manhã |

In [ ]:
antes = antes.assign(
    tem_cappuccino=(antes["nome_produto"] == "Cappuccino").astype(int),
    eh_ifood=(antes["canal"] == "iFood").astype(int),
    eh_manha=(antes["periodo"] == "Manhã").astype(int),
)

por_pedido = antes.groupby(["id_cliente", "id_venda"]).agg(
    valor=("valor_total", "sum"),
    cappuccino=("tem_cappuccino", "max"),
    ifood=("eh_ifood", "max"),
    manha=("eh_manha", "max"),
).reset_index()

features = por_pedido.groupby("id_cliente").agg(
    pedidos=("id_venda", "count"),
    gasto_total=("valor", "sum"),
    pct_cappuccino=("cappuccino", "mean"),
    pct_ifood=("ifood", "mean"),
    pct_manha=("manha", "mean"),
).reset_index()
features["ticket_medio"] = features["gasto_total"] / features["pedidos"]
features.head()

Agora juntamos com os dados cadastrais (bairro, idade, fidelidade) e criamos a **resposta** que o modelo deve aprender: `sumiu` (1 = sim, 0 = não).

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
base = features.merge(clientes, on="___", how="left")
base["sumiu"] = (~base["id_cliente"].isin(clientes_depois)).astype(int)
base["fidelidade"] = (base["programa_fidelidade"] == "Sim").astype(int)

base[["id_cliente", "nome", "bairro", "pedidos", "pct_cappuccino", "sumiu"]].head()

Uma olhada rápida antes de treinar: qual bairro perdeu mais clientes?

In [ ]:
(base.groupby("bairro")["sumiu"].mean() * 100).sort_values().plot(
    kind="barh", color="#1FD3F0", title="% de clientes que sumiram por bairro")
plt.show()

## 3. Preparar os dados para o modelo

O modelo só entende números, e o bairro é texto. Na Prática 2, a análise mostrou que os bairros da **orla** (Tambaú, Manaíra e Cabo Branco) foram os que mais perderam clientes. Vamos usar essa descoberta para criar uma feature nova: `mora_na_orla` (1 = sim, 0 = não).

💡 Usar o conhecimento do negócio para criar features se chama **feature engineering**, e é uma das habilidades mais valiosas de um cientista de dados.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
base["mora_na_orla"] = base["bairro"].isin(["Tambaú", "Manaíra", "___"]).astype(int)

colunas = ["pedidos", "gasto_total", "ticket_medio", "pct_cappuccino", "pct_ifood", "pct_manha", "idade", "fidelidade", "mora_na_orla"]
X = base[colunas].copy()
X["idade"] = X["idade"].fillna(X["idade"].median())
y = base["sumiu"]
X.head()

## 4. Separar treino e teste

Se o modelo for avaliado com os mesmos dados que usou para aprender, ele pode só ter **decorado** as respostas. Por isso, separamos:

- **75% para treino**: o modelo aprende com esses clientes.
- **25% para teste**: clientes que o modelo nunca viu, para conferir se ele aprendeu de verdade.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=___, random_state=42, stratify=y)
print("Treino:", len(X_treino), "clientes | Teste:", len(X_teste), "clientes")

## 5. Treinar o modelo: Árvore de Decisão 🌳

Uma árvore de decisão aprende uma sequência de perguntas do tipo "o cliente mora na orla?", "gasta pouco?". É um dos modelos mais fáceis de entender.

O `class_weight="balanced"` diz ao modelo para dar a mesma atenção a quem ficou e a quem sumiu, já que queremos encontrar o máximo possível de clientes em risco.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
modelo = DecisionTreeClassifier(max_depth=3, min_samples_leaf=15, class_weight="balanced", random_state=42)
modelo.___(X_treino, y_treino)
print("Modelo treinado! 🎉")

## 6. O modelo é bom?

Comparamos com um "chute": dizer que **ninguém** vai sumir. Se o modelo não for melhor que isso, ele não serve.

Também medimos o **recall**: dos clientes que realmente sumiram, quantos o modelo conseguiu identificar. Para uma campanha de reconquista, essa é a métrica que mais importa.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
previsoes = modelo.___(X_teste)

chute = [0] * len(y_teste)
print(f"Acurácia do chute 'ninguém some': {accuracy_score(y_teste, chute):.0%}")
print(f"Acurácia do modelo:               {accuracy_score(y_teste, previsoes):.0%}")
print(f"Recall do modelo:                 {recall_score(y_teste, previsoes):.0%} dos clientes que sumiram foram identificados")

## 7. Abrindo a caixa: o que o modelo aprendeu?

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(modelo, feature_names=X.columns, class_names=["Ficou", "Sumiu"], filled=True, rounded=True, fontsize=10, impurity=False)
plt.show()

In [ ]:
importancia = pd.Series(modelo.feature_importances_, index=X.columns)
importancia[importancia > 0].sort_values().plot(kind="barh", color="#3DF29A", title="O que mais pesa para um cliente sumir")
plt.show()

🤔 **Para pensar:** as features mais importantes batem com o que a análise encontrou? Morar na orla, perto do concorrente novo, aumenta o risco. E quem gasta pouco é um cliente menos fiel, que some com mais facilidade.

## 8. Da previsão à ação: a lista da campanha

Um modelo só gera valor quando vira **decisão**. Vamos calcular a probabilidade de cada cliente sumir e montar a lista dos que mais precisam de atenção, entre os que **ainda estão comprando**.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
base["risco"] = modelo.predict_proba(X)[:, 1]

ainda_ativos = base[base["sumiu"] == ___]
campanha = ainda_ativos.sort_values("risco", ascending=False).head(15)
campanha[["nome", "bairro", "pedidos", "pct_cappuccino", "pct_ifood", "risco"]].round(2)

## ⭐ Desafio bônus

Nenhum modelo acerta 100%: as pessoas param de comprar por motivos que não estão nos dados (mudaram de cidade, entraram de dieta...). O importante é ser **bem melhor que o chute** e gerar uma ação útil.

Mude o `max_depth` da árvore (tente 2, 5 e 10) e rode de novo a partir do passo 5. O que acontece com a acurácia no teste? Uma árvore muito profunda começa a **decorar** os dados de treino (isso se chama **overfitting**).

## 🎯 O que a Ciência de Dados entregou

Em vez de só explicar o passado, agora o dono do café tem **uma lista de clientes para ligar ou mandar um cupom esta semana**.

📋 Envie no formulário quais foram as features mais importantes do seu modelo para ganhar seu cupom do sorteio! 🎁